# Phase 2 confirmation: diagnose, replicate, then ablate

This notebook is the executable continuation of the completed Phase 2 seed-0 pilot. It restores all persistent artifacts, enforces the Phase 1 gate, audits delay semantics, re-evaluates the frozen seed-0 actors on untouched seeds, trains only seeds 1 and 2 for the three primary methods, runs a pass-needed cooperation probe, and evaluates a predeclared seed-level gate. The no-action-delay ablation runs only if the replicated data authorize it. Every training, evaluation, trace, and video command is synced to Drive immediately.

## 1. Initialize Colab and restore the persistent workspace

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

REPO_URL = "https://github.com/djdhillxn/football"
REPO_DIR = Path("/content/robot-soccer-transfer")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT = DRIVE_MOUNT / "MyDrive" / "RobotSoccerTransfer"

def checked(command, label, cwd=None):
    print("+", " ".join(map(str, command)))
    result = subprocess.run([str(value) for value in command], cwd=cwd, check=False)
    if result.returncode != 0:
        raise RuntimeError(f"{label} failed with status {result.returncode}")
    return result

checked([sys.executable, "--version"], "Python version")
subprocess.run(["nvidia-smi"], check=False)
from google.colab import drive
drive.mount(str(DRIVE_MOUNT))
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)
os.environ["ROBOSOCCER_DRIVE_PROJECT"] = str(DRIVE_PROJECT)

if REPO_DIR.exists():
    dirty = subprocess.run(["git", "status", "--porcelain"], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
    if dirty:
        raise RuntimeError("The Colab checkout has local changes; preserve or discard them before pulling.")
    checked(["git", "fetch", "origin", "--quiet"], "git fetch", REPO_DIR)
    checked(["git", "pull", "--ff-only"], "git pull", REPO_DIR)
else:
    checked(["git", "clone", REPO_URL, str(REPO_DIR)], "git clone")
os.chdir(REPO_DIR)

# Preserve Colab's CUDA-enabled Torch/NumPy/pandas stack; install project-specific packages only.
checked([sys.executable, "-m", "pip", "install", "-q", "gymnasium==1.2.1", "pettingzoo==1.26.1", "pymunk==7.3.0", "PyYAML==6.0.3", "tensorboard==2.20.0", "imageio==2.37.2", "imageio-ffmpeg==0.6.0", "pytest==8.4.2", "ruff==0.14.5"], "dependency install", REPO_DIR)
checked([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"], "editable install", REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
import torch, robosoccer
print("commit", subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip())
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "project", robosoccer.__version__)

In [ ]:
from robosoccer.artifacts import (
    read_run_metadata, sync_artifacts_from_drive, sync_auxiliary_artifacts_to_drive,
    sync_run_to_drive, sync_workspace_to_drive,
)
from robosoccer.config import load_config
from robosoccer.evaluation import phase1_readiness_audit

RUNS_ROOT = REPO_DIR / "runs"
PRIMARY_METHODS = ["mappo_nominal", "mappo_uniform_dr", "mappo_failure_dr"]
PRIMARY_CONFIGS = {name: REPO_DIR / "configs" / f"{name}.yaml" for name in PRIMARY_METHODS}

def completed_run(experiment, seed, required_disabled_parameters=None):
    candidates = []
    for path in RUNS_ROOT.iterdir() if RUNS_ROOT.is_dir() else []:
        metadata = read_run_metadata(path) if path.is_dir() else None
        if metadata and metadata.get("status") == "complete" and metadata.get("experiment_name") == experiment and int(metadata.get("seed", -1)) == int(seed):
            if required_disabled_parameters is not None:
                resolved = load_config(path / "resolved_config.yaml")
                if resolved["randomization"].get("disabled_parameters", []) != required_disabled_parameters:
                    continue
            candidates.append(path)
    if not candidates:
        raise FileNotFoundError(f"No complete {experiment} seed {seed} run was found")
    return max(candidates, key=lambda path: path.name).resolve()

def latest_run(experiment):
    candidates = []
    for path in RUNS_ROOT.iterdir() if RUNS_ROOT.is_dir() else []:
        metadata = read_run_metadata(path) if path.is_dir() else None
        if metadata and metadata.get("status") == "complete" and metadata.get("experiment_name") == experiment:
            candidates.append(path)
    if not candidates:
        raise FileNotFoundError(f"No complete {experiment} run was found")
    return max(candidates, key=lambda path: path.name).resolve()

def run_and_sync(command, run_dir, label):
    print(f"\n[{label}]", " ".join(map(str, command)))
    result = subprocess.run([str(value) for value in command], cwd=REPO_DIR, check=False)
    sync_result = sync_run_to_drive(run_dir, DRIVE_PROJECT)
    print(json.dumps(sync_result, indent=2))
    if result.returncode != 0:
        raise RuntimeError(f"{label} failed with status {result.returncode}; available diagnostics were saved to Drive")

def train_if_missing(experiment, config_path, seed, overrides=None, required_disabled_parameters=None):
    try:
        run_dir = completed_run(experiment, seed, required_disabled_parameters)
        print(f"reuse {experiment} seed {seed}: {run_dir.name}")
        return run_dir
    except FileNotFoundError:
        pass
    before = {path.name for path in RUNS_ROOT.iterdir()} if RUNS_ROOT.is_dir() else set()
    command = [sys.executable, "-m", "scripts.train", "--config", str(config_path), "--seed", str(seed)]
    if experiment != config_path.stem:
        command.extend(["--run-name", experiment])
    command.extend(overrides or [])
    print(f"\n[train {experiment} seed {seed}]", " ".join(command))
    result = subprocess.run(command, cwd=REPO_DIR, check=False)
    after = {path.name for path in RUNS_ROOT.iterdir()} if RUNS_ROOT.is_dir() else set()
    created = sorted(after - before)
    synced = []
    for name in created:
        path = RUNS_ROOT / name
        metadata = read_run_metadata(path) if path.is_dir() else None
        if metadata and metadata.get("status") in {"complete", "failed"}:
            print(json.dumps(sync_run_to_drive(path, DRIVE_PROJECT), indent=2))
            synced.append(path)
    if result.returncode != 0:
        raise RuntimeError(f"Training failed for {experiment} seed {seed}; diagnostics from {len(synced)} run(s) were saved")
    return completed_run(experiment, seed, required_disabled_parameters)

def run_auxiliary(command, label, include_reports=True):
    print(f"\n[{label}]", " ".join(map(str, command)))
    result = subprocess.run([str(value) for value in command], cwd=REPO_DIR, check=False)
    print(json.dumps(sync_auxiliary_artifacts_to_drive(DRIVE_PROJECT, REPO_DIR, include_reports), indent=2))
    if result.returncode != 0:
        raise RuntimeError(f"{label} failed with status {result.returncode}; partial artifacts were saved")

In [ ]:
# A fresh Colab runtime needs checkpoints as well as lightweight analysis files.
drive_sync = sync_artifacts_from_drive(DRIVE_PROJECT, REPO_DIR, include_training_artifacts=True)
print(json.dumps(drive_sync, indent=2))

baseline_run = latest_run("base")
ippo_run = latest_run("ippo_nominal")
phase1_audit = phase1_readiness_audit(
    load_config(REPO_DIR / "configs" / "base.yaml"),
    json.loads((baseline_run / "eval" / "baseline_summary.json").read_text()),
    json.loads((ippo_run / "eval" / "abstract_standard" / "summary.json").read_text()),
    json.loads((ippo_run / "eval" / "pymunk_transfer" / "summary.json").read_text()),
)
print(json.dumps(phase1_audit, indent=2))
if not phase1_audit["phase2_ready"]:
    raise RuntimeError("PHASE 2 NO-GO: the restored Phase 1 gate does not pass")
print("PHASE 2 GO: restored Phase 1 evidence passes every safeguard.")

## 2. Freeze and audit the protocol before any new training

This is the hard stop for code quality and delay semantics. The audit checks latencies 0–5 in both simulators, FIFO reset behavior, and the four physics repeats per macro action, and writes hand-checkable traces.

In [ ]:
checked(["ruff", "check", "robosoccer", "scripts", "tests"], "ruff", REPO_DIR)
checked([sys.executable, "-m", "pytest", "-q"], "pytest", REPO_DIR)
AUDIT_DIR = RUNS_ROOT / "logs" / "phase2_protocol"
checked([sys.executable, "-m", "scripts.audit_action_delay", "--config", "configs/base.yaml", "--output-dir", str(AUDIT_DIR), "--maximum-latency", "5"], "action-delay audit", REPO_DIR)
delay_audit = json.loads((AUDIT_DIR / "action_delay_audit.json").read_text())
print(json.dumps(delay_audit, indent=2))
if not delay_audit["passed"]:
    raise RuntimeError("Delay semantics audit failed; do not train")
print(json.dumps(sync_auxiliary_artifacts_to_drive(DRIVE_PROJECT, REPO_DIR, include_reports=False), indent=2))

In [ ]:
protocol_config = load_config(REPO_DIR / "configs" / "base.yaml")
seeds = protocol_config["evaluation"]["seed_bases"]
assert seeds["confirmatory_abstract_test"] == 230000
assert seeds["confirmatory_transfer_test"] == 240000
assert seeds["confirmatory_profile_test"] == 241000
assert seeds["confirmatory_robustness_test"] == 242000
assert seeds["cooperation_probe"] == 250000
assert protocol_config["evaluation"]["replication_gate"]["minimum_training_seeds"] == 3
core_diff = subprocess.run(["git", "diff", "--quiet", "--", "configs", "robosoccer", "scripts", "tests"], cwd=REPO_DIR, check=False)
if core_diff.returncode != 0:
    raise RuntimeError("Core protocol files differ from the checked-out commit; commit or revert before training")
PROTOCOL_COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
print("Frozen confirmation commit:", PROTOCOL_COMMIT)

## 3. Restore seed 0 and train only the two missing replication seeds

In [ ]:
# Seed 0 is the frozen pilot actor. It is re-evaluated, never extended.
primary_runs = {(method, 0): completed_run(method, 0) for method in PRIMARY_METHODS}
for key, path in primary_runs.items():
    print(key, path.name)

In [ ]:
# Three independent seed-1 jobs; each finished or failed run is saved before continuing.
for method in PRIMARY_METHODS:
    primary_runs[(method, 1)] = train_if_missing(method, PRIMARY_CONFIGS[method], 1)

In [ ]:
# Three independent seed-2 jobs; each finished or failed run is saved before continuing.
for method in PRIMARY_METHODS:
    primary_runs[(method, 2)] = train_if_missing(method, PRIMARY_CONFIGS[method], 2)

## 4. Evaluate every frozen actor on untouched confirmatory suites

The exact suite name is retained in every row. Abstract success for transfer gaps means intercept-defender success only; profile means and delay–noise grid AUC remain separate outcomes.

In [ ]:
def evaluate_if_missing(run_dir, prefix, simulator, suite, seed, episodes=None):
    summary_path = run_dir / "eval" / prefix / "summary.json"
    if summary_path.is_file():
        summary = json.loads(summary_path.read_text())
        if int(summary.get("episode_count", 0)) > 0:
            print("reuse", run_dir.name, prefix, "episodes", summary["episode_count"])
            return summary
    command = [sys.executable, "-m", "scripts.evaluate", "--run-dir", str(run_dir), "--checkpoint", "best", "--simulator", simulator, "--suite", suite, "--seed", str(seed), "--prefix", prefix]
    if episodes is not None:
        command.extend(["--episodes", str(episodes)])
    run_and_sync(command, run_dir, f"{run_dir.name} {prefix}")
    return json.loads(summary_path.read_text())

def evaluate_confirmation(run_dir):
    evaluate_if_missing(run_dir, "confirmatory_abstract_standard", "abstract", "standard", seeds["confirmatory_abstract_test"], 100)
    evaluate_if_missing(run_dir, "confirmatory_pymunk_transfer", "pymunk", "transfer", seeds["confirmatory_transfer_test"], 100)
    evaluate_if_missing(run_dir, "confirmatory_pymunk_profiles", "pymunk", "profiles", seeds["confirmatory_profile_test"], 50)
    evaluate_if_missing(run_dir, "confirmatory_pymunk_robustness", "pymunk", "robustness", seeds["confirmatory_robustness_test"])
    evaluate_if_missing(run_dir, "confirmatory_pymunk_cooperation", "pymunk", "cooperation", seeds["cooperation_probe"], 100)


In [ ]:
# Re-evaluate all three frozen seed-0 pilot actors.
for method in PRIMARY_METHODS:
    evaluate_confirmation(primary_runs[(method, 0)])

In [ ]:
for method in PRIMARY_METHODS:
    evaluate_confirmation(primary_runs[(method, 1)])

In [ ]:
for method in PRIMARY_METHODS:
    evaluate_confirmation(primary_runs[(method, 2)])

## 5. Matched delay traces and pass-needed videos for the three pilot actors

These are diagnostic artifacts, not cherry-picked evidence. Every method receives the same episode seeds; matched video mode records exactly those seeds rather than searching for preferred outcomes.

In [ ]:
for method in PRIMARY_METHODS:
    run_dir = primary_runs[(method, 0)]
    trace_summary = run_dir / "eval" / "confirmatory_delay_traces" / "summary.json"
    if not trace_summary.is_file():
        run_and_sync([sys.executable, "-m", "scripts.trace_action_delays", "--run-dir", str(run_dir), "--simulator", "pymunk", "--delays", "0", "1", "2", "3", "4", "5", "--seed", str(seeds["delay_audit"]), "--prefix", "confirmatory_delay_traces"], run_dir, f"{method} matched delay traces")
    videos = list((run_dir / "videos").glob(f"{method}_pymunk_cooperation_seed*.mp4"))
    if len(videos) < 3:
        run_and_sync([sys.executable, "-m", "scripts.record_video", "--run-dir", str(run_dir), "--checkpoint", "best", "--simulator", "pymunk", "--episodes", "3", "--scenario", "cooperation", "--matched", "--seed", str(seeds["cooperation_probe"])], run_dir, f"{method} matched cooperation videos")

## 6. Build suite-safe comparisons and enforce the seed-level gate

In [ ]:
def compare_protocol(extra_runs=None):
    run_paths = [primary_runs[(method, seed)] for seed in [0, 1, 2] for method in PRIMARY_METHODS]
    run_paths.extend(extra_runs or [])
    command = [sys.executable, "-m", "scripts.compare_runs", "--phase", "final", "--export-report", "--runs", *map(str, run_paths)]
    run_auxiliary(command, "suite-safe comparison")
    comparison_dir = sorted((RUNS_ROOT / "comparisons").glob("*_final"))[-1]
    return comparison_dir, json.loads((comparison_dir / "replication_summary.json").read_text())

comparison_dir, replication = compare_protocol()
print(json.dumps(replication, indent=2))
from IPython.display import Image as DisplayImage, display
import pandas as pd
display(pd.read_csv(comparison_dir / "replication_per_seed.csv"))
display(pd.read_csv(comparison_dir / "suite_comparison.csv"))
for figure in [comparison_dir / "method_success_by_suite.png", comparison_dir / "canonical_transfer_gap.png"]:
    if figure.is_file():
        display(DisplayImage(filename=str(figure)))
print("CONFIRMATION GATE:", "PASS" if replication["confirmation_passed"] else "DOES NOT PASS")
print("DELAY ABLATION:", "AUTHORIZED" if replication["delay_ablation_authorized"] else "NOT AUTHORIZED")

## 7. Conditional targeted delay ablation

This cell does nothing unless the profile advantage and delay failure both replicate under the locked gate. When authorized, it changes exactly one training mechanism: action latency is neutralized even inside combined profiles. Evaluation still restores the full delay challenge.

In [ ]:
ablation_runs = []
if replication["delay_ablation_authorized"]:
    for training_seed in [0, 1, 2]:
        run_dir = train_if_missing(
            "mappo_failure_no_action_delay",
            REPO_DIR / "configs" / "mappo_failure_dr.yaml",
            training_seed,
            ["randomization.disabled_parameters=[action_latency]"],
            required_disabled_parameters=["action_latency"],
        )
        ablation_runs.append(run_dir)
        evaluate_confirmation(run_dir)
    comparison_dir, replication = compare_protocol(ablation_runs)
    print("Targeted delay ablation completed and compared:", comparison_dir)
else:
    print("Skipped by design. The replicated evidence does not authorize a delay ablation.")

## 8. Compile reports when LaTeX is available and perform the final safety sync

In [ ]:
if shutil.which("latexmk"):
    checked(["latexmk", "-pdf", "-interaction=nonstopmode", "-halt-on-error", "-outdir=reports", "reports/main.tex"], "main report build", REPO_DIR)
    checked(["latexmk", "-pdf", "-interaction=nonstopmode", "-halt-on-error", "-outdir=reports", "reports/surrogate_notes.tex"], "surrogate report build", REPO_DIR)
else:
    print("latexmk is not installed in this runtime; report sources and generated tables are still saved.")
print(json.dumps(sync_workspace_to_drive(DRIVE_PROJECT, REPO_DIR), indent=2))
print("Phase 2 confirmation workflow finished. Read the replication gate before making any harder-simulator decision.")